# Dimensionality reduction with TICA: the alanine pentapeptide

Principal component analysis (PCA) finds the directions of largest *variance* in feature space, which are not necessarily the *slowest* ones. Time-lagged independent component analysis (TICA) instead finds the linear combinations of features that decorrelate most slowly, i.e. the best linear approximation to the slow dynamical modes. Here we test the `DimRed.doTICA` method on the alanine pentapeptide, using the same data as in the [featurization example](ala_pentapeptide_featurize.ipynb), and compare it with PCA.

The checks we carry out are:
1. the implied timescales of the TICA components should converge with the lag time and should not exceed the slowest relaxation of the system;
2. an MSM built on a TICA space should recover the timescales of the TICA components.

In [ ]:
%load_ext autoreload
%autoreload 2
import os, io, contextlib
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
sns.set(style="ticks", color_codes=True, font_scale=1.2)
sns.set_style({"xtick.direction": "in", "ytick.direction": "in"})

We download the trajectory from the [OSF repository](https://osf.io/a2vc7/) if it is not already present.

In [ ]:
from mastermsm.test.download_data import download_osf_ala5, _DATA_DIR
download_osf_ala5()

### Featurization
We load the trajectory and use the $\sin$ and $\cos$ of the backbone torsions as features, which are then standardized (whitened).

In [ ]:
from mastermsm.trajectory import traj
tr = traj.TimeSeries(top=os.path.join(_DATA_DIR, 'ala5.gro'),
                     xtc=os.path.join(_DATA_DIR, 'ala5.xtc'))
print(tr.mdt, "\ndt =", tr.dt, "ps")
X = traj.Featurizer(tr)
X.add_torsions(sincos=True)
DR = traj.DimRed(tr)
DR.whiten()
features0 = tr.features.copy()   # keep a pristine copy, since DimRed methods overwrite tr.features
print(features0.shape)

### Choosing the lag time
`doTICA` takes the lag time **in frames**. As with MSMs, the implied timescales $t_i = -\tau/\ln\lambda_i$ of the TICA components should be insensitive to the lag time $\tau$ once $\tau$ is long enough. We scan $\tau$ (restoring the features each time, since the method replaces them with the projection).

In [ ]:
lags = [1, 2, 5, 10, 20, 50, 100, 200, 500, 1000]
its = []
for lag in lags:
    tr.features = features0.copy()
    with contextlib.redirect_stdout(io.StringIO()):
        DR.doTICA(lag=lag, n=4)
    its.append(DR.tica_timescales[:6])
its = np.array(its)

fig, ax = plt.subplots()
for i in range(its.shape[1]):
    ax.plot(np.array(lags)*tr.dt, its[:, i], 'o-', ms=6, label=str(i+1))
ax.fill_between(np.array(lags)*tr.dt, 1, np.array(lags)*tr.dt, color='lightgray')
ax.set_xlabel(r'Lag time $\tau$ [ps]')
ax.set_ylabel(r'$t_i$ [ps]')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_ylim(1e2, 3e4)
ax.legend(title='TIC', fontsize=9, ncol=2)
plt.tight_layout()

Unlike in an MSM, the linear TICA timescales are *not* converged at short lag times: they increase with $\tau$ and the slowest one only approaches a plateau of about 10 ns for $\tau$ of a few ns. This is expected, since a linear combination of torsion features is only an approximation to the slow mode, and implied timescales from TICA are therefore lower bounds that converge as $\tau$ grows. At much longer lags than shown here the estimate becomes dominated by statistical noise, because few frame pairs remain in a 500 ns trajectory. We choose $\tau$ = 200 frames (2 ns), where the slowest timescale is close to its plateau value.

In [ ]:
lag = 200
tr.features = features0.copy()
DR.doTICA(lag=lag, n=4)
tica_features = tr.features.copy()

### PCA versus TICA
We project the same data onto the first two components from each method. TICA resolves several distinct metastable basins in its first two coordinates, whereas the PCA projection, whose directions are set by variance, is a largely featureless distribution.

In [ ]:
tr.features = features0.copy()
DR.doPCA(n=4)
pca_features = tr.features.copy()

fig, ax = plt.subplots(1, 2, figsize=(8, 3.6), sharey=False)
for a, f, name in zip(ax, [pca_features, tica_features], ['PC', 'TIC']):
    a.hist2d(f[:, 0], f[:, 1], bins=[60, 60], cmap='Blues',
             norm=mpl.colors.LogNorm())
    a.set_xlabel(name + '1'); a.set_ylabel(name + '2')
ax[0].set_title('PCA'); ax[1].set_title('TICA, lag = %g ps' % (lag*tr.dt))
plt.tight_layout()

The first TIC is the slowest-decorrelating coordinate, so its autocorrelation at the lag time should equal the corresponding eigenvalue. This is a direct check of the implementation on real data.

In [ ]:
ls = np.arange(1, 401, 4)
def autocorr(x, lag):
    x = x - x.mean()
    return np.mean(x[:-lag]*x[lag:])/np.var(x)

fig, ax = plt.subplots()
for i in range(3):
    ax.plot(ls*tr.dt, [autocorr(tica_features[:, i], l) for l in ls], label='TIC%i' % (i+1))
for i in range(2):
    ax.plot(ls*tr.dt, [autocorr(pca_features[:, i], l) for l in ls], '--', label='PC%i' % (i+1))
ax.set_xlabel('Time [ps]'); ax.set_ylabel('Autocorrelation')
ax.legend(fontsize=9)
plt.tight_layout()
print("Autocorrelation of TIC at lag:", [round(autocorr(tica_features[:, i], lag), 3) for i in range(4)])
print("TICA eigenvalues             :", np.round(DR.tica_eigenvalues[:4], 3))

### Discretization and MSM on the TICA space
Finally we cluster in the 4-dimensional TICA space and in the 4-dimensional PCA space, with the same number of $k$-means centers, and build an MSM in each case. We enforce symmetrization of the count matrix (`sym=True`), consistent with the reversibility assumed in TICA; without it, statistical noise can produce complex-conjugate eigenvalue pairs, which show up as spurious duplicated timescales. The implied timescales from the MSM can be compared against the TICA estimates.

In [ ]:
from mastermsm.msm import msm

def build_msm(features, k=100, seed=0):
    np.random.seed(seed)
    tr.features = features.copy()
    DISC = traj.Discretizer(tr)
    DISC.kmeans(k=k, dim=4)
    tr.find_keys()
    sm = msm.SuperMSM([tr], sym=True)
    sm.convergence_test(time=[10, 20, 50, 100, 200, 500, 1000], error=False)
    return sm

sm_tica = build_msm(tica_features)
sm_pca = build_msm(pca_features)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9, 3.6), sharey=True)
for a, sm, name in zip(ax, [sm_tica, sm_pca], ['TICA', 'PCA']):
    for i in range(4):
        d = np.array([[x, sm.msms[x].tauT[i]] for x in sorted(sm.msms.keys())])
        a.plot(d[:, 0], d[:, 1], 'o-', ms=6)
    a.fill_between(d[:, 0], 1, d[:, 0], color='lightgray')
    a.set_xscale('log'); a.set_yscale('log')
    a.set_xlabel(r'$\Delta t$ [ps]'); a.set_title('MSM on ' + name + ' space')
ax[0].set_ylabel(r'$\tau_i$ [ps]')
ax[0].set_ylim(1e2, 3e4)
plt.tight_layout()

The table compares the first relaxation times from the TICA eigenvalue problem ($\tau$ = 2 ns) with those of the MSM in TICA space ($\Delta t$ = 100 ps). The slowest timescale should agree well; for the faster modes TICA, being linear, resolves less of the dynamics, so the agreement is only approximate.

In [ ]:
tica_ts = DR.tica_timescales[:4]
msm_ts = sm_tica.msms[100].tauT[:4]
print("%-4s %12s %12s" % ("i", "TICA [ps]", "MSM [ps]"))
for i in range(4):
    print("%-4i %12.0f %12.0f" % (i+1, tica_ts[i], msm_ts[i]))